# Anomaly & Outlier Detection

Companion notebook for the [Anomaly Detection lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/14-anomaly-detection).

We score anomalies three ways — a **z-score** statistical test, **kNN distance**, and a tiny
**Isolation-Forest**-style path length — then show why **accuracy is the wrong metric** under
extreme imbalance and how the **threshold** trades precision against recall. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — Data: mostly normal, a few anomalies

A dense normal cluster plus a handful of scattered anomalies — the typical extreme imbalance.

In [ ]:
normal = rng.normal(0, 1, size=(490, 2))
anom = rng.uniform(-6, 6, size=(10, 2))
X = np.vstack([normal, anom])
y = np.r_[np.zeros(len(normal)), np.ones(len(anom))]    # 1 = anomaly (2% of data)
print(f'{int(y.sum())} anomalies out of {len(X)} points ({100*y.mean():.0f}%)')

## 2 — Three anomaly scores

z-score (distance from the mean in std units), kNN distance (to the k-th neighbor), and a simplified
isolation score (how few random splits isolate a point).

In [ ]:
def zscore_anomaly(X):
    return np.linalg.norm((X - X.mean(0)) / X.std(0), axis=1)

def knn_anomaly(X, k=5):
    D = np.linalg.norm(X[:, None] - X[None, :], axis=2)
    D.sort(axis=1)
    return D[:, k]                                       # distance to the k-th neighbor

def isolation_score(X, n_trees=100, seed=0):
    r = np.random.default_rng(seed)
    n = len(X)
    depths = np.zeros(n)
    for _ in range(n_trees):
        idx = np.arange(n); depth = np.zeros(n); active = np.ones(n, bool)
        lo, hi = X.min(0), X.max(0)
        bounds = {tuple(idx): (lo.copy(), hi.copy())}
        # simplified: split the whole space repeatedly, count splits until each point is alone-ish
        for d in range(1, 12):
            f = r.integers(0, X.shape[1])
            t = r.uniform(X[active, f].min(), X[active, f].max()) if active.sum() > 1 else 0
            left = active & (X[:, f] < t)
            # points in the smaller partition are 'more isolated' -> stop deepening them
            small = left if left.sum() < (active & ~left).sum() else (active & ~left)
            depth[small & (depth == 0)] = d
            active = active & ~small
            if active.sum() <= 1:
                depth[active & (depth == 0)] = d
                break
        depths += depth
    return -depths / n_trees                              # short path (small depth) -> high anomaly

for name, score in [('z-score', zscore_anomaly(X)), ('kNN', knn_anomaly(X)), ('isolation', isolation_score(X))]:
    top10 = set(np.argsort(-score)[:10])
    recall = len(top10 & set(np.where(y == 1)[0])) / 10
    print(f'{name:10s}: recall@10 = {recall:.1f}')

## 3 — Accuracy lies; use precision/recall at a threshold

A detector that calls everything 'normal' is 98% accurate here yet useless. We threshold the z-score
and trace precision vs recall as the cutoff moves.

In [ ]:
print('accuracy of a "call everything normal" detector:', f'{(y==0).mean():.3f}  <- useless!')
score = zscore_anomaly(X)
print('\nthreshold   precision   recall   #flagged')
for thr in np.percentile(score, [90, 95, 98, 99]):
    flag = score >= thr
    prec = (flag & (y==1)).sum() / max(flag.sum(), 1)
    rec = (flag & (y==1)).sum() / (y==1).sum()
    print(f'{thr:7.2f}     {prec:.2f}        {rec:.2f}      {flag.sum()}')
print('\nLower threshold -> higher recall but more false alarms. Pick by your alert budget.')

## ✏️ Your turn

**Exercise.** Implement `zscore(X)` (per-point Euclidean norm of the standardized features) and
`recall_at_alert_rate(score, y, rate)` — flag the top `rate` fraction of points by score and return
the recall (fraction of true anomalies caught). This is how you evaluate a detector under a fixed
alert budget.

In [ ]:
def zscore(X):
    # TODO(you): standardize features, return the Euclidean norm per point
    return ...

def recall_at_alert_rate(score, y, rate):
    # TODO(you): flag the top `rate` fraction by score; return recall over the true anomalies (y==1)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
s = zscore(X)
assert np.allclose(s, zscore_anomaly(X))
# flagging more points can only help recall (monotonic)
assert recall_at_alert_rate(s, y, 0.10) >= recall_at_alert_rate(s, y, 0.02)
assert 0.0 <= recall_at_alert_rate(s, y, 0.05) <= 1.0
# z-score catches most of these far-flung anomalies even at a 5% alert rate
assert recall_at_alert_rate(s, y, 0.05) >= 0.7
print('\u2713 zscore and recall@alert-rate are correct')

<details>
<summary>Solution</summary>

```python
def zscore(X):
    return np.linalg.norm((X - X.mean(0)) / X.std(0), axis=1)

def recall_at_alert_rate(score, y, rate):
    k = max(1, int(rate * len(score)))
    flagged = set(np.argsort(-score)[:k])
    caught = len(flagged & set(np.where(y == 1)[0]))
    return caught / (y == 1).sum()
```

Reporting recall at a fixed alert rate (or precision@k) is the honest way to score an anomaly
detector — it bakes in the real constraint that the team can only investigate so many alerts.

</details>